# Comparative Study of Deep Learning Models for Glacier & Glacial Lake Mapping

This notebook trains and compares **three state-of-the-art semantic segmentation models** on the Kaggle 410 Glacial Dataset (`archive.zip`):
1. **U-Net** (Baseline Convolutional Encoder-Decoder)
2. **SUNet** (Swin-UNet / Spectrally-Augmented Swin Transformer)
3. **Attention U-Net** (U-Net with Attention Gates on skip connections)

---

## 1. Environment & GPU Setup
Verifies CUDA GPU acceleration in Google Colab (**Runtime -> Change runtime type -> Hardware accelerator -> GPU**).

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For faster training, switch to GPU in Runtime -> Change runtime type.")

## 2. Unzip Archive Dataset (`archive.zip`)
Extracts `archive.zip` containing 410 glacial images and corresponding binary masks.

In [ ]:
import zipfile

dataset_dir = "./glacial-lake-dataset"
if not os.path.exists(dataset_dir):
    if os.path.exists("archive.zip"):
        print("Unzipping archive.zip...")
        with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
            zip_ref.extractall("./")
        print("Extraction complete!")
    else:
        print("Please upload 'archive.zip' to the Colab files sidebar before running this cell.")
else:
    print(f"Dataset directory '{dataset_dir}' found!")

images_dir = os.path.join(dataset_dir, "images")
masks_dir = os.path.join(dataset_dir, "masks")

if os.path.exists(images_dir) and os.path.exists(masks_dir):
    img_files = sorted([f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
    print(f"Total Glacial Images found: {len(img_files)}")
else:
    print("Error: Images or Masks folder not found.")

## 3. PyTorch Dataset Loader & Train/Val Split
Splits 410 pairs into **80% Training** (328 samples) and **20% Validation** (82 samples).

In [ ]:
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

class GlacialDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_files):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_files = img_files
        self.img_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor()
        ])
        self.mask_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        fname = self.img_files[idx]
        img_path = os.path.join(self.images_dir, fname)
        mask_path = os.path.join(self.masks_dir, fname)

        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        img_tensor = self.img_transform(img)
        mask_tensor = (self.mask_transform(mask) > 0.5).float()
        return img_tensor, mask_tensor

full_dataset = GlacialDataset(images_dir, masks_dir, img_files)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

torch.manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Training Dataset: {len(train_dataset)} images")
print(f"Validation Dataset: {len(val_dataset)} images")

## 4. Model Definitions (U-Net, Attention U-Net, SUNet)
Defines the 3 comparative semantic segmentation architectures (`in_channels=3`, `out_channels=1`).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(True)
        )
    def forward(self, x): return self.conv(x)

# 1. Standard U-Net
class UNet(nn.Module):
    def __init__(self, in_c=3, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        d1 = self.cup1(torch.cat([self.up1(x5), x4], 1))
        d2 = self.cup2(torch.cat([self.up2(d1), x3], 1))
        d3 = self.cup3(torch.cat([self.up3(d2), x2], 1))
        d4 = self.cup4(torch.cat([self.up4(d3), x1], 1))
        return self.outc(d4)

# 2. Attention U-Net
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.Wx = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        psi_conv = nn.Conv2d(F_int, 1, 1)
        nn.init.constant_(psi_conv.bias, 1.0)
        self.psi = nn.Sequential(psi_conv, nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(True)
    def forward(self, g, x):
        g1 = self.Wg(g)
        x1 = self.Wx(x)
        if g1.shape[2:] != x1.shape[2:]: g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=True)
        return x * self.psi(self.relu(g1 + x1))

class AttentionUNet(nn.Module):
    def __init__(self, in_c=3, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        self.ag1 = AttentionGate(512, 512, 256)
        self.ag2 = AttentionGate(256, 256, 128)
        self.ag3 = AttentionGate(128, 128, 64)
        self.ag4 = AttentionGate(64, 64, 32)
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        u1 = self.up1(x5)
        d1 = self.cup1(torch.cat([u1, self.ag1(u1, x4)], 1))
        u2 = self.up2(d1)
        d2 = self.cup2(torch.cat([u2, self.ag2(u2, x3)], 1))
        u3 = self.up3(d2)
        d3 = self.cup3(torch.cat([u3, self.ag3(u3, x2)], 1))
        u4 = self.up4(d3)
        d4 = self.cup4(torch.cat([u4, self.ag4(u4, x1)], 1))
        return self.outc(d4)

# 3. SUNet (Swin-UNet)
class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        B, C, H, W = x.shape
        xf = x.flatten(2).transpose(1, 2)
        att, _ = self.attn(self.norm1(xf), self.norm1(xf), self.norm1(xf))
        xf = xf + att
        xf = xf + self.mlp(self.norm2(xf))
        return xf.transpose(1, 2).view(B, C, H, W)

class SUNet(nn.Module):
    def __init__(self, in_c=3, out_c=1):
        super().__init__()
        self.inc = DoubleConv(in_c, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024), SwinBlock(1024, 8))
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.cup1 = nn.Sequential(DoubleConv(1024, 512), SwinBlock(512, 4))
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.cup2 = nn.Sequential(DoubleConv(512, 256), SwinBlock(256, 4))
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.cup3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.cup4 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_c, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        d1 = self.cup1(torch.cat([self.up1(x5), x4], 1))
        d2 = self.cup2(torch.cat([self.up2(d1), x3], 1))
        d3 = self.cup3(torch.cat([self.up3(d2), x2], 1))
        d4 = self.cup4(torch.cat([self.up4(d3), x1], 1))
        return self.outc(d4)

print("All 3 Model Architectures (UNet, AttentionUNet, SUNet) initialized for 3-channel input.")

## 5. Model Training & Validation Benchmark Loop
Trains U-Net, Attention U-Net, and SUNet for 25 epochs each on GPU using Hybrid BCE + Dice Loss.

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        return 1.0 - (2.0 * (p * t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

class HybridLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return self.alpha * self.bce(logits, targets) + (1 - self.alpha) * self.dice(logits, targets)

def evaluate_model(model, loader, device, th=0.50):
    model.eval()
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = (torch.sigmoid(model(x)) >= th).float()
            tp += (preds * y).sum().item()
            fp += (preds * (1 - y)).sum().item()
            fn += ((1 - preds) * y).sum().item()
            tn += ((1 - preds) * (1 - y)).sum().item()
    eps = 1e-8
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    return dice, iou, prec, rec

models_dict = {
    "U-Net (Baseline)": UNet(in_c=3, out_c=1),
    "Attention U-Net": AttentionUNet(in_c=3, out_c=1),
    "SUNet (Swin-UNet)": SUNet(in_c=3, out_c=1)
}

benchmark_results = []
trained_models = {}
epochs = 25
lr = 2e-4
os.makedirs("results", exist_ok=True)

for name, m in models_dict.items():
    print(f"\n===================================================")
    print(f"   Training Model: {name} ({epochs} Epochs)")
    print(f"===================================================")
    m = m.to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    crit = HybridLoss(alpha=0.5)
    best_dice = 0.0
    best_metrics = (0, 0, 0, 0)
    for epoch in range(1, epochs + 1):
        m.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            loss = crit(m(bx), by)
            loss.backward()
            opt.step()
            train_loss += loss.item()
        d, i, p, r = evaluate_model(m, val_loader, device)
        if d > best_dice:
            best_dice = d
            best_metrics = (d, i, p, r)
            torch.save(m.state_dict(), f"results/{name.lower().replace(' ', '_')}_best.pth")
        if epoch % 5 == 0 or epoch == epochs:
            print(f"   Epoch {epoch:02d}/{epochs} -> Dice: {d:.4f} | IoU: {i:.4f} | Precision: {p:.4f} | Recall: {r:.4f}")
    
    trained_models[name] = m
    params_m = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
    benchmark_results.append({
        "Model Architecture": name,
        "Parameters (M)": round(params_m, 2),
        "Dice Score": round(best_metrics[0], 4),
        "IoU Score": round(best_metrics[1], 4),
        "Precision": round(best_metrics[2], 4),
        "Recall": round(best_metrics[3], 4)
    })

df_results = pd.DataFrame(benchmark_results)
print("\n===================================================")
print("         FINAL COMPARATIVE STUDY BENCHMARK          ")
print("===================================================")
display(df_results)

# Save Results CSV
df_results.to_csv("results/comparative_metrics.csv", index=False)
print("Saved results table to 'results/comparative_metrics.csv'")

## 6. Save & Plot Visual Segmentation Overlay Comparisons

In [ ]:
# Plot Metrics Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_results["Model Architecture"]))
width = 0.2

ax.bar(x - 1.5*width, df_results["Dice Score"], width, label="Dice Score", color='#1f77b4')
ax.bar(x - 0.5*width, df_results["IoU Score"], width, label="IoU Score", color='#2ca02c')
ax.bar(x + 0.5*width, df_results["Precision"], width, label="Precision", color='#ff7f0e')
ax.bar(x + 1.5*width, df_results["Recall"], width, label="Recall", color='#d62728')

ax.set_ylabel("Score")
ax.set_title("Comparative Model Performance (410 Glacial Dataset)")
ax.set_xticks(x)
ax.set_xticklabels(df_results["Model Architecture"], fontweight="bold")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("results/comparative_metrics_chart.png", dpi=300)
plt.show()

# Visual Overlay Comparisons
sample_patch, sample_mask = val_dataset[0]
bx = sample_patch.unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
rgb = sample_patch.permute(1, 2, 0).numpy()
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

axes[0].imshow(rgb)
axes[0].set_title("Input Image")
axes[0].axis('off')

axes[1].imshow(sample_mask.squeeze().numpy(), cmap='gray')
axes[1].set_title("Ground Truth Mask")
axes[1].axis('off')

for idx, (name, m) in enumerate(trained_models.items()):
    m.eval()
    with torch.no_grad():
        prob = torch.sigmoid(m(bx)).squeeze().cpu().numpy()
        pred = (prob >= 0.50).astype(np.uint8)
    axes[2 + idx].imshow(pred, cmap='Blues')
    d_val = df_results.loc[df_results['Model Architecture'] == name, 'Dice Score'].values[0]
    axes[2 + idx].set_title(f"{name}\n(Dice: {d_val:.4f})")
    axes[2 + idx].axis('off')

plt.tight_layout()
plt.savefig("results/comparative_predictions_overlay.png", dpi=300)
plt.show()

## 7. Automatic Download of Trained Models & Results
Triggers automatic browser download of metrics CSV, prediction overlay charts, and best model weights.

In [ ]:
try:
    from google.colab import files
    print("Downloading results CSV, charts, and model checkpoints...")
    files.download("results/comparative_metrics.csv")
    files.download("results/comparative_metrics_chart.png")
    files.download("results/comparative_predictions_overlay.png")
    files.download("results/u-net_(baseline)_best.pth")
    files.download("results/attention_u-net_best.pth")
    files.download("results/sunet_(swin-unet)_best.pth")
except Exception as e:
    print(f"Local environment detected or browser download skipped: {e}")